In [1]:
import cv2
import mediapipe as mp
import numpy as np
import random
import time

from mediapipe.tasks.python import vision
from mediapipe.tasks.python import BaseOptions

# Load model
model_path = "hand_landmarker.task"

options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

# Score
user_score = 0
comp_score = 0

round_active = False
start_time = 0

user_move = "None"
computer_move = ""
result_text = ""

# ---------- Functions ----------
def fingers_up(hand):
    return [
        hand[8].y < hand[6].y,
        hand[12].y < hand[10].y,
        hand[16].y < hand[14].y,
        hand[20].y < hand[18].y
    ]

def get_user_move(count):
    if count == 0:
        return "Rock"
    elif count == 2:
        return "Scissors"
    elif count >= 4:
        return "Paper"
    else:
        return "None"

def get_winner(user, comp):
    global user_score, comp_score

    if user == comp:
        return "Draw"
    elif (user == "Rock" and comp == "Scissors") or \
         (user == "Scissors" and comp == "Paper") or \
         (user == "Paper" and comp == "Rock"):
        user_score += 1
        return "You Win 🎉"
    else:
        comp_score += 1
        return "Computer Wins 🤖"

# ---------- Game Loop ----------
while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    count = 0

    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            fingers = fingers_up(hand)
            count = fingers.count(True)

    # -------- Countdown Logic --------
    if round_active:
        elapsed = time.time() - start_time

        if elapsed < 1:
            text = "3"
        elif elapsed < 2:
            text = "2"
        elif elapsed < 3:
            text = "1"
        elif elapsed < 4:
            text = "SHOW"
        else:
            # Capture move
            user_move = get_user_move(count)
            computer_move = random.choice(["Rock", "Paper", "Scissors"])

            if user_move != "None":
                result_text = get_winner(user_move, computer_move)

            round_active = False

        cv2.putText(frame, text, (250,250),
                    cv2.FONT_HERSHEY_SIMPLEX, 4, (0,255,255), 5)

    # -------- UI Display --------
    cv2.putText(frame, f'You: {user_score}', (10,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.putText(frame, f'Computer: {comp_score}', (400,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.putText(frame, f'Your Move: {user_move}', (10,100),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    cv2.putText(frame, f'Computer: {computer_move}', (10,150),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

    cv2.putText(frame, f'Result: {result_text}', (10,200),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    cv2.putText(frame, "Press SPACE to Play", (150,400),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    cv2.imshow("🔥 Rock Paper Scissors AI PRO", frame)

    key = cv2.waitKey(1) & 0xFF

    # Start round
    if key == 32 and not round_active:  # SPACE
        round_active = True
        start_time = time.time()
        result_text = ""
        user_move = "None"

    # Exit
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()